# FOG 检测模型训练 — Yi & Hwang (2025) CNN+GRU+ECA

基于 Daphnet 公开数据集的迁移学习训练
输出: ~44KB TFLite 模型，直接部署到 Flutter App

In [ ]:
# @title 1. 安装依赖
!pip install -q tensorflow tensorflow-datasets scikit-learn matplotlib
!pip install -q gdown  # 用于下载 Daphnet 数据集


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import os, json, gdown, zipfile

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')


In [ ]:
# @title 2. 下载并加载 Daphnet 数据集
# Daphnet FOG 数据集: https://archive.ics.uci.edu/dataset/245/daphnet+freezing+of+gait

# 从 UCI 下载
!wget -q https://archive.ics.uci.edu/static/public/245/daphnet+freezing+of+gait.zip -O daphnet.zip
!unzip -q daphnet.zip -d daphnet/

# 或者如果已有预处理的 numpy 文件，从 Google Drive 加载
# 这里提供预处理脚本

def load_daphnet(data_dir='daphnet/'):
    """加载 Daphnet 原始数据，分割为滑动窗口"""
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.txt')])
    print(f'发现 {len(files)} 个患者文件: {files}')
    
    all_X, all_y = [], []
    window_size = 128  # ~2秒 @ 64Hz
    step = 32  # 滑动步长
    
    for f in files:
        data = np.loadtxt(os.path.join(data_dir, f))
        # 列: accelX, accelY, accelZ, label (0=normal, 1=FOG)
        X = data[:, :3]  # 只用 3 轴加速度
        y = data[:, -1]
        
        # 滑动窗口
        for i in range(0, len(X) - window_size, step):
            win_X = X[i:i+window_size]
            win_y = y[i:i+window_size]
            # 标签：窗口内 > 50% 为 FOG 则标记为 FOG
            label = 1 if np.mean(win_y) > 0.5 else 0
            all_X.append(win_X)
            all_y.append(label)
    
    return np.array(all_X), np.array(all_y)

# 加载
X_raw, y_raw = load_daphnet()
print(f'原始数据: X={X_raw.shape}, y={y_raw.shape}')
print(f'FOG 占比: {y_raw.mean():.1%}')


In [ ]:
# @title 3. 数据预处理
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

# 标准化（每轴独立）
X_reshaped = X_raw.reshape(-1, 3)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped).reshape(X_raw.shape)

# 增加通道维度 (samples, time, channels, 1) 适配 Conv2D
X = X_scaled[..., np.newaxis]  # -> (samples, 128, 3, 1)
y = y_raw

# 留一法划分 (Leave-One-Subject-Out)
# 简单起见: 前 8 人训练，后 2 人验证
# 实际上需要按患者分组，这里假设文件顺序对应患者
n_subjects = 10
samples_per_subject = len(X) // n_subjects
train_end = 8 * samples_per_subject

X_train, X_val = X[:train_end], X[train_end:]
y_train, y_val = y[:train_end], y[train_end:]

print(f'训练集: {X_train.shape}, FOG={y_train.mean():.1%}')
print(f'验证集: {X_val.shape}, FOG={y_val.mean():.1%}')

# 类别权重（缓解 FOG 样本少的问题）
neg_weight = 1.0 / (1 - y_train.mean())
pos_weight = 1.0 / y_train.mean()
class_weight = {0: neg_weight, 1: pos_weight}
print(f'类别权重: normal={neg_weight:.1f}, FOG={pos_weight:.1f}')


In [ ]:
# @title 4. 构建 Yi & Hwang 模型架构

def build_fog_model(input_shape=(128, 3, 1)):
    """
    Yi & Hwang (2025) CNN + GRU + Residual + ECA Attention
    参数量: ~14K
    """
    inputs = keras.Input(shape=input_shape)
    
    # ── CNN 前端 ──
    # Conv1: 32 filters, kernel 12
    x = layers.Conv2D(32, kernel_size=(12, 1), padding='same',
                       kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=(2, 1))(x)
    x = layers.Dropout(0.5)(x)
    
    # Conv2: 64 filters, kernel 12
    x = layers.Conv2D(64, kernel_size=(12, 1), padding='same',
                       kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=(2, 1))(x)
    x = layers.Dropout(0.5)(x)
    
    # 展平时间维度，保持通道
    # 输出形状: (batch, channels, features) -> 置换为 (batch, time, channels)
    shape = x.shape
    x = layers.Reshape((shape[1], -1))(x)
    
    # ── GRU + Residual ──
    gru_out = layers.Bidirectional(layers.GRU(64, return_sequences=True))(x)
    # Residual: 如果维度不匹配，先投影
    if gru_out.shape[-1] != x.shape[-1]:
        x_proj = layers.Dense(gru_out.shape[-1])(x)
        x = layers.Add()([gru_out, x_proj])
    else:
        x = layers.Add()([gru_out, x])
    x = layers.LayerNormalization()(x)
    
    # ── ECA (Efficient Channel Attention) ──
    # 全局平均池化 -> 1D 卷积跨通道 -> sigmoid -> 重标定
    gap = layers.GlobalAveragePooling1D()(x)  # (batch, channels)
    gap = layers.Reshape((-1, 1))(gap)  # (batch, channels, 1)
    
    # 自适应 1D 卷积核大小
    k = int(np.ceil(gap.shape[1] / 4))
    if k % 2 == 0:
        k += 1
    k = max(3, min(k, gap.shape[1]))  # clamp
    
    attn = layers.Conv1D(1, kernel_size=k, padding='same',
                         use_bias=False)(gap)  # (batch, channels, 1)
    attn = layers.Activation('sigmoid')(attn)
    attn = layers.Reshape((-1,))(attn)  # (batch, channels)
    
    # 重标定
    x = layers.Multiply()([x, layers.Reshape((1, -1))(attn)])
    
    # ── 分类头 ──
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = keras.Model(inputs, outputs, name='FoG_Detector')
    return model


model = build_fog_model()
model.summary()

total_params = model.count_params()
print(f'\n总参数量: {total_params:,} ({total_params/1024:.1f}K)')


In [ ]:
# @title 5. 训练

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall(),
             keras.metrics.AUC(name='auc')]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=15, mode='max',
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(history.history['auc'], label='train')
axes[1].plot(history.history['val_auc'], label='val')
axes[1].set_title('AUC')
axes[1].legend()
plt.show()


In [ ]:
# @title 6. 评估

y_pred_prob = model.predict(X_val, verbose=0).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_val, y_pred, target_names=['Normal', 'FOG']))
print(f'F1: {f1_score(y_val, y_pred):.4f}')

# 混淆矩阵
cm = confusion_matrix(y_val, y_pred)
print('\nConfusion Matrix:')
print(f'            Pred Normal  Pred FOG')
print(f'Actual Normal   {cm[0,0]:5d}      {cm[0,1]:5d}')
print(f'Actual FOG      {cm[1,0]:5d}      {cm[1,1]:5d}')


In [ ]:
# @title 7. 导出 TFLite（量化至 44KB）

# ① 先转标准 TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS  # 部分操作需要
]
tflite_model = converter.convert()
print(f'标准 TFLite: {len(tflite_model)/1024:.1f} KB')

# ② 动态范围量化（推荐，平衡大小与精度）
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
tflite_quant = converter.convert()
print(f'量化 TFLite: {len(tflite_quant)/1024:.1f} KB')

# ③ 保存
os.makedirs('models', exist_ok=True)
with open('models/fog_detector.tflite', 'wb') as f:
    f.write(tflite_quant)
print(f'\n已保存: models/fog_detector.tflite')

# 验证 TFLite 推理
interpreter = tf.lite.Interpreter(model_content=tflite_quant)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'\nTFLite 输入: {input_details[0]["shape"]}')
print(f'TFLite 输出: {output_details[0]["shape"]}')

# 测一次推理延迟
import time
test_input = X_val[:1].astype(np.float32)
start = time.perf_counter()
for _ in range(100):
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
elapsed = (time.perf_counter() - start) / 100
print(f'单次推理延迟: {elapsed*1000:.2f} ms')


In [ ]:
# @title 8. 下载模型文件
from google.colab import files

# 下载到本地
files.download('models/fog_detector.tflite')

# 也保存一份标准化参数供 Flutter 端使用
scaler_params = {
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist(),
    'window_size': 128,
    'sample_rate': 64,  # Daphnet 原始采样率
}
with open('models/scaler_params.json', 'w') as f:
    json.dump(scaler_params, f)
files.download('models/scaler_params.json')

print('\n✅ 完成！将 fog_detector.tflite 放入 Flutter 项目的 assets/models/')
print('   将 scaler_params.json 也一同放入供 App 端预处理使用')
